# Marco 1 — Bronze | SRAG 2019–2026 (sem API CKAN)

**Objetivo:** consultar os CSVs oficiais de SRAG, preservá-los na camada Bronze e gerar diagnósticos sem modificar os arquivos originais.

**Fonte:** Portal de Dados Abertos do SUS — SRAG 2019 a 2026. As URLs de download foram registradas a partir dos recursos oficiais e são consultadas diretamente com Python + `requests`.

**Sem API CKAN:** este notebook **não consulta endpoints de catálogo CKAN** nem procura novas versões automaticamente. Se o portal publicar arquivos novos, atualize manualmente a URL do ano correspondente após verificá-la na página oficial.

**Segurança:** download desativado por padrão, limites de armazenamento, arquivos temporários, SHA-256 e política de não sobrescrever arquivos existentes. As rotinas de diagnóstico trabalham com os CSVs locais sem transformá-los.


## 1. Bibliotecas e pastas

Localizamos automaticamente a raiz do repositório, independentemente de o Jupyter iniciar na pasta `notebooks/` ou na raiz do projeto. O `requirements.txt` e a pasta `notebooks/` ajudam a identificá-la.

In [35]:
import csv
import hashlib
import json
import os
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from IPython.display import display

RAIZ = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()),
    None,
)
if RAIZ is None:
    raise RuntimeError("Abra o repositório no VS Code antes de executar o notebook.")

BRONZE = RAIZ / "data" / "bronze"
METADATA = RAIZ / "metadata"
BRONZE.mkdir(parents=True, exist_ok=True)
METADATA.mkdir(parents=True, exist_ok=True)

MIB = 1024**2
GIB = 1024**3

print("Raiz:", RAIZ)
print("Bronze:", BRONZE)
print(f"Livre no disco: {shutil.disk_usage(BRONZE).free / GIB:.2f} GiB")

Raiz: /home/roso/Projetos/R0Z01-Projeto-3-Projeto-integrador---Group-IV
Bronze: /home/roso/Projetos/R0Z01-Projeto-3-Projeto-integrador---Group-IV/data/bronze
Livre no disco: 341.12 GiB


## 2. Parâmetros e limites

Os limites foram dimensionados após o inventário inicial (aprox. 4,69 GiB para os oito CSVs; confira o espaço livre novamente antes de transferir). **Nada é baixado sem autorização explícita.**

A lista de endereços oficiais é mantida manualmente na seção 3. Não são feitas consultas a APIs de catálogo.


In [48]:
ANOS = list(range(2019, 2027))

LIMITE_ARQUIVO = 2 * GIB      # Cada CSV pode ocupar até 2 GiB
LIMITE_BRONZE = 6 * GIB      # Máximo somado entre CSVs e temporários
RESERVA_DISCO = 30 * GIB    # Espaço livre mínimo a preservar

AUTORIZAR_DOWNLOAD = True
ANOS_PARA_BAIXAR = [2021, 2022, 2023, 2024, 2025, 2026]  # Anos que serão baixados

PORTAL = "https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026"
TIMEOUT_HTTP = (10, 45)

## 3. Fontes oficiais de SRAG por ano

O dicionário `FONTES_REFERENCIA` contém uma URL de download por ano, **verificada no inventário atual**. Os anos 2019–2024 utilizam arquivos datados de 23/03/2026; os anos 2025–2026 utilizam arquivos datados de 16/11/2026.

Para atualizar um endereço no futuro:

1. Abra a [página oficial de SRAG 2019–2026](https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026).
2. Verifique a URL e o ano do recurso CSV, sem usar APIs de catálogo.
3. Atualize **somente** a entrada do ano em `FONTES_REFERENCIA` e refaça o inventário (`HEAD`).

**Atenção:** o trecho `ckan.saude.gov.br` no *caminho do objeto S3* é apenas parte do endereço de hospedagem do arquivo. Isso **não** é uma requisição à API CKAN.


In [49]:
# URLs de recursos CSV SRAG publicados no portal oficial do SUS.
# Atualize manualmente a entrada do ano quando o portal publicar outra versão.
FONTES_REFERENCIA = {
    ano: (
        f"https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/{ano}/"
        f"INFLUD{str(ano)[-2:]}-{('23-03-2026' if ano <= 2024 else '16-11-2026')}.csv"
    )
    for ano in ANOS
}


def url_csv_srag(url):
    """Aceita somente URLs HTTPS com o padrão esperado do objeto CSV SRAG oficial."""
    if not isinstance(url, str):
        return False
    p = urlparse(url)
    return (
        p.scheme == "https"
        and p.hostname == "s3.sa-east-1.amazonaws.com"
        and p.query == ""
        and p.fragment == ""
        and re.fullmatch(
            r"/ckan\.saude\.gov\.br/SRAG/20\d{2}/INFLUD\d{2}-\d{2}-\d{2}-\d{4}\.csv",
            p.path,
        ) is not None
    )


# A origem dos dados é fixa: CSVs oficiais, sem chamadas a API de catálogo.
fontes = dict(FONTES_REFERENCIA)
origem_fontes = "URLs oficiais de CSV SRAG verificadas manualmente"
for ano, url in fontes.items():
    if not url_csv_srag(url):
        raise ValueError(f"URL SRAG inválida para {ano}: {url}")
    nome = Path(urlparse(url).path).name
    if not nome.startswith(f"INFLUD{ano % 100:02d}-"):
        raise ValueError(f"Ano incompatível entre URL e CSV: {ano}")

print("Origem dos links:", origem_fontes)
for ano, url in fontes.items():
    print(ano, url.rsplit("/", 1)[-1])


Origem dos links: URLs oficiais de CSV SRAG verificadas manualmente
2019 INFLUD19-23-03-2026.csv
2020 INFLUD20-23-03-2026.csv
2021 INFLUD21-23-03-2026.csv
2022 INFLUD22-23-03-2026.csv
2023 INFLUD23-23-03-2026.csv
2024 INFLUD24-23-03-2026.csv
2025 INFLUD25-16-11-2026.csv
2026 INFLUD26-16-11-2026.csv


## 4. Inventário: consultar o tamanho de todos os arquivos sem baixá-los

Enviamos requisições **HEAD** para os oito arquivos. Se o servidor não informar o tamanho, ele será apresentado como `desconhecido`; por segurança, o download fica bloqueado até uma verificação manual.

Nenhum CSV é gravado nesta etapa.

In [50]:
def consultar_tamanho(ano, url):
    try:
        with requests.head(
            url,
            timeout=TIMEOUT_HTTP,
            allow_redirects=True,
            headers={"Accept-Encoding": "identity"},
        ) as resposta:
            resposta.raise_for_status()
            cabecalho = resposta.headers.get("Content-Length")
            tamanho = int(cabecalho) if cabecalho and cabecalho.isdecimal() else None
            return {
                "ano": ano, "url": url, "http": resposta.status_code,
                "bytes": tamanho,
                "MiB": round(tamanho / MIB, 2) if tamanho is not None else None,
                "etag": resposta.headers.get("ETag"),
                "ultima_modificacao": resposta.headers.get("Last-Modified"),
                "situacao": "OK" if tamanho is not None else "tamanho desconhecido",
            }
    except (requests.RequestException, ValueError) as erro:
        return {
            "ano": ano, "url": url, "http": None, "bytes": None,
            "MiB": None, "etag": None, "ultima_modificacao": None,
            "situacao": f"falha de consulta: {str(erro)[:100]}",
        }


inventario = {ano: consultar_tamanho(ano, url) for ano, url in fontes.items()}
quadro = pd.DataFrame(inventario.values())
display(quadro[["ano", "http", "MiB", "situacao"]])

conhecidos = [item["bytes"] for item in inventario.values() if item["bytes"] is not None]
print(f"Total informado: {sum(conhecidos) / GIB:.2f} GiB ",
      f"({len(conhecidos)} de {len(inventario)} anos consultados)")
print(f"Espaço livre atual: {shutil.disk_usage(BRONZE).free / GIB:.2f} GiB")
if len(conhecidos) < len(inventario):
    print("ATENÇÃO: o total é parcial; algum ano está sem tamanho informado.")
else:
    print("Tamanhos confirmados para todos os anos inventariados.")

,ano,http,MiB,situacao
0,2019,200,49.27,OK
1,2020,200,1201.39,OK
2,2021,200,1740.95,OK
3,2022,200,594.16,OK
4,2023,200,301.57,OK
5,2024,200,288.24,OK
6,2025,200,364.42,OK
7,2026,200,266.40,OK


Total informado: 4.69 GiB  (8 de 8 anos consultados)
Espaço livre atual: 340.93 GiB
Tamanhos confirmados para todos os anos inventariados.


## 5. Funções de integridade e download reutilizável

- Calcula SHA-256 para cada arquivo (não confundir com ETag).
- Compara `ETag` do `HEAD` com o `GET` **quando ambos existirem**. Uma mudança interrompe a transferência antes da escrita. Não é uma prova criptográfica de integridade. <https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/ETag>
- Preserva todas as versões do CSV; não sobrescreve arquivos existentes.
- Produz metadados de duas formas: `bronze_ANO.json` (atalho para versão mais recente verificada) e `bronze_ANO_NOME_DO_ARQUIVO.json` (histórico por versão).
- Aceita `fonte` e `data_verificacao` de arquivos antigos ao **ler**, mas grava o formato atual.
- Interrompe downloads ao alcançar cotas de arquivo, Bronze ou reserva do disco.

In [51]:
def uso_bronze_bytes():
    """Conta CSVs e temporários locais para conferir a cota da Bronze."""
    return sum(
        p.stat().st_size for p in BRONZE.iterdir()
        if p.is_file() and (p.name.endswith(".csv") or p.name.endswith(".part"))
    )


def calcular_sha256(caminho):
    h = hashlib.sha256()
    with caminho.open("rb") as entrada:
        for bloco in iter(lambda: entrada.read(MIB), b""):
            h.update(bloco)
    return h.hexdigest()


def salvar_json_atomico(caminho, conteudo):
    temporario = caminho.with_suffix(caminho.suffix + ".part")
    try:
        with temporario.open("w", encoding="utf-8") as saida:
            json.dump(conteudo, saida, ensure_ascii=False, indent=2)
        os.replace(temporario, caminho)
    finally:
        temporario.unlink(missing_ok=True)


def normalizar_metadados(meta):
    """Aceita nomes antigos de 2019 sem perder dados no formato atual."""
    meta = dict(meta or {})
    meta["schema_versao"] = 2
    meta["url"] = meta.get("url") or meta.get("fonte")
    meta["verificado_em_utc"] = meta.get("verificado_em_utc") or meta.get("data_verificacao")
    meta.setdefault("coletado_em_utc", None)
    meta.pop("fonte", None)
    meta.pop("data_verificacao", None)
    return meta


def arquivo_metadados_versao(ano, nome):
    # O nome vem de URL validada; Path(...).stem mantém o histórico legível.
    return METADATA / f"bronze_{ano}_{Path(nome).stem}.json"


def registrar_metadados(ano, nome, meta):
    """Persiste a versão atual e preserva metadados anteriores ao atualizar o apontador anual."""
    meta = normalizar_metadados(meta)
    meta["schema_versao"] = 2
    atual = METADATA / f"bronze_{ano}.json"
    if atual.exists():
        antigo = normalizar_metadados(json.loads(atual.read_text(encoding="utf-8")))
        nome_antigo = antigo.get("arquivo")
        if nome_antigo and nome_antigo != nome:
            arquivo_antigo = arquivo_metadados_versao(ano, nome_antigo)
            if not arquivo_antigo.exists():
                salvar_json_atomico(arquivo_antigo, antigo)
    salvar_json_atomico(arquivo_metadados_versao(ano, nome), meta)
    salvar_json_atomico(atual, meta)


def obter_metadados_anteriores(ano, nome):
    """Prefere metadados da versão; aceita o JSON anual legado somente se apontar ao mesmo CSV."""
    for caminho in [arquivo_metadados_versao(ano, nome), METADATA / f"bronze_{ano}.json"]:
        if caminho.exists():
            meta = normalizar_metadados(json.loads(caminho.read_text(encoding="utf-8")))
            if meta.get("arquivo") == nome:
                return meta
    return {}


def baixar_arquivo(ano, *, fontes=fontes, inventario=inventario):
    """Coleta um arquivo para qualquer ano, sem sobrescrever versões locais."""
    if ano not in fontes:
        raise ValueError(f"Ano fora do catálogo: {ano}")
    url = fontes[ano]
    info = inventario.get(ano, {})
    tamanho = info.get("bytes")
    if not url_csv_srag(url):
        raise RuntimeError(f"{ano}: URL não corresponde ao padrão de CSV oficial esperado")
    if info.get("url") != url or info.get("http") != 200 or not isinstance(tamanho, int) or tamanho <= 0:
        raise RuntimeError(f"{ano}: refaça o inventário HEAD para esta URL antes de baixar")
    nome = Path(urlparse(url).path).name
    if not nome.startswith(f"INFLUD{ano % 100:02d}-"):
        raise RuntimeError(f"{ano}: nome de arquivo com ano incompatível")

    destino_arquivo = BRONZE / nome
    parcial = BRONZE / f"{nome}.part"

    if destino_arquivo.exists():
        local_bytes = destino_arquivo.stat().st_size
        if local_bytes != tamanho:
            raise RuntimeError(f"{ano}: tamanho local não coincide com HEAD; verificar a versão")
        sha = calcular_sha256(destino_arquivo)
        anterior = obter_metadados_anteriores(ano, nome)
        if anterior.get("sha256") and anterior["sha256"] != sha:
            raise RuntimeError(f"{ano}: SHA-256 local mudou desde o último registro")
        if anterior.get("etag") and info.get("etag") and anterior["etag"] != info["etag"]:
            raise RuntimeError(f"{ano}: ETag remoto mudou; não sobrescreva a versão local")
        agora = datetime.now(timezone.utc).isoformat()
        meta = {
            "schema_versao": 2, "ano": ano, "url": url, "arquivo": nome,
            "tamanho_bytes": local_bytes, "sha256": sha, "etag": info.get("etag"),
            "ultima_modificacao": info.get("ultima_modificacao"),
            "coletado_em_utc": anterior.get("coletado_em_utc"),
            "verificado_em_utc": agora,
            "origem": "arquivo local preexistente; tamanho comparado ao HEAD",
        }
        registrar_metadados(ano, nome, meta)
        print(f"{ano}: arquivo já existe; não foi baixado novamente")
        return destino_arquivo

    if parcial.exists():
        raise RuntimeError(f"{ano}: temporário {parcial.name} já existe; confira-o antes de remover")
    if tamanho > LIMITE_ARQUIVO:
        raise RuntimeError(f"{ano}: tamanho de {tamanho / MIB:.1f} MiB excede a cota por arquivo")
    ocupado = uso_bronze_bytes()
    if ocupado + tamanho > LIMITE_BRONZE:
        raise RuntimeError(f"{ano}: a transferência ultrapassaria a cota total Bronze")
    if shutil.disk_usage(BRONZE).free - tamanho < RESERVA_DISCO:
        raise RuntimeError(f"{ano}: a transferência ultrapassaria a reserva mínima do disco")

    transferidos = 0
    sha = hashlib.sha256()
    try:
        with requests.get(url, stream=True, timeout=TIMEOUT_HTTP,
                          headers={"Accept-Encoding": "identity"}) as resposta:
            resposta.raise_for_status()
            if resposta.status_code != 200:
                raise RuntimeError(f"{ano}: GET inesperado HTTP {resposta.status_code}")
            etag_head, etag_get = info.get("etag"), resposta.headers.get("ETag")
            if etag_head and etag_get and etag_head != etag_get:
                raise RuntimeError(f"{ano}: o ETag mudou entre HEAD e GET; refaça o inventário")
            resposta_tamanho = resposta.headers.get("Content-Length")
            if resposta_tamanho and resposta_tamanho.isdecimal() and int(resposta_tamanho) != tamanho:
                raise RuntimeError(f"{ano}: tamanho mudou entre HEAD e GET")
            if resposta.headers.get("Content-Encoding", "identity").lower() not in ("identity", ""):
                raise RuntimeError(f"{ano}: codificação de transferência inesperada")

            with parcial.open("xb") as saida:
                for bloco in resposta.iter_content(chunk_size=MIB):
                    if not bloco:
                        continue
                    proximo = transferidos + len(bloco)
                    if proximo > tamanho or proximo > LIMITE_ARQUIVO:
                        raise RuntimeError(f"{ano}: tamanho transferido excedeu o limite")
                    if ocupado + proximo > LIMITE_BRONZE:
                        raise RuntimeError(f"{ano}: cota Bronze atingida")
                    if shutil.disk_usage(BRONZE).free - len(bloco) < RESERVA_DISCO:
                        raise RuntimeError(f"{ano}: reserva de espaço atingida")
                    saida.write(bloco)
                    sha.update(bloco)
                    transferidos = proximo
                    if transferidos // (50 * MIB) > (transferidos - len(bloco)) // (50 * MIB):
                        print(f"{ano}: {transferidos / MIB:.1f} MiB recebidos")
        if transferidos != tamanho:
            raise RuntimeError(f"{ano}: arquivo incompleto, {transferidos} de {tamanho} bytes")
        os.replace(parcial, destino_arquivo)
    finally:
        # Se houver interrupção tratada, remove o temporário desta tentativa.
        parcial.unlink(missing_ok=True)

    meta = {
        "schema_versao": 2, "ano": ano, "url": url, "arquivo": nome,
        "tamanho_bytes": transferidos, "sha256": sha.hexdigest(),
        "etag": etag_get or etag_head,
        "ultima_modificacao": resposta.headers.get("Last-Modified") or info.get("ultima_modificacao"),
        "coletado_em_utc": datetime.now(timezone.utc).isoformat(),
        "verificado_em_utc": datetime.now(timezone.utc).isoformat(),
        "origem": "download novo; tamanho confirmado pelo HEAD/GET",
    }
    registrar_metadados(ano, nome, meta)
    print(f"{ano}: CSV salvo em {destino_arquivo} ({transferidos / MIB:.1f} MiB)")
    return destino_arquivo

## 6. Executar a coleta somente com autorização explícita

**Antes de mudar para `True`:** confira o inventário da seção 4, aumente as cotas com cautela se necessário e selecione os anos. O código para no primeiro erro; não fará downloads silenciosos em sequência.

Você pode começar por `ANOS_PARA_BAIXAR = [2019]` — como o arquivo de 2019 já existe no seu computador, apenas será conferido. Depois tente `[2020]`. Para todos os anos, configure `list(range(2019, 2027))`, **somente após verificar os tamanhos**.

In [ ]:
if not AUTORIZAR_DOWNLOAD:
    print("🔒 Download desativado. Inventário e código podem ser testados sem transferir CSVs.")
else:
    for ano in ANOS_PARA_BAIXAR:
        print(f"\n--- Coleta de {ano} ---")
        baixar_arquivo(ano)
    print("Coleta selecionada finalizada.")

## 7. Diagnóstico de estrutura e campos ausentes (opcional, um ano por vez)

Esta etapa **não modifica** o CSV original. A escolha da versão local é controlada: prefere o arquivo da lista de URLs; depois, um arquivo explicitamente informado; em seguida, um arquivo registrado em metadados ou um único candidato local. Se houver várias opções ambíguas, pede uma escolha explícita.

- Verifica os registros com `csv.reader(strict=True)`, conferindo o número de campos antes de usar Pandas.
- O Pandas usa `on_bad_lines="error"`, `skip_blank_lines=False` e leitura em blocos de 5.000 registros.
- A codificação padrão é **latin-1 por escolha documentada**, com detecção de BOM UTF-8. Sem BOM não há identificação infalível de encoding.
- O relatório `qualidade_ANO.json` é um diagnóstico agregado, não uma transformação Bronze.


In [56]:
ANO_DIAGNOSTICO = 2024
NOME_ARQUIVO_DIAGNOSTICO = None    # Ex.: "INFLUD19-23-03-2026.csv"; obrigatório se houver ambiguidade.
CODIFICACAO_PADRAO = "latin-1"     # Escolha documentada para as bases CSV SRAG verificadas.
CAMPOS_INTERESSE = [
    "NU_NOTIFIC", "DT_NOTIFIC", "DT_SIN_PRI", "CS_SEXO",
    "SG_UF_NOT", "CLASSI_FIN", "EVOLUCAO", "HOSPITAL", "UTI",
]


def selecionar_csv_local(ano, *, nome_arquivo=None, fontes=fontes):
    candidatos = sorted(BRONZE.glob(f"INFLUD{ano % 100:02d}-*.csv"))
    if not candidatos:
        raise FileNotFoundError(f"{ano}: nenhum CSV encontrado em {BRONZE}")
    if nome_arquivo is not None:
        escolhido = BRONZE / nome_arquivo
        if escolhido not in candidatos:
            raise ValueError(f"{ano}: versão escolhida não encontrada entre os arquivos locais")
        return escolhido

    url = fontes.get(ano)
    if url and url_csv_srag(url):
        esperado = BRONZE / Path(urlparse(url).path).name
        if esperado in candidatos:
            return esperado

    meta = METADATA / f"bronze_{ano}.json"
    if meta.exists():
        dados = normalizar_metadados(json.loads(meta.read_text(encoding="utf-8")))
        nome = dados.get("arquivo")
        candidato_meta = BRONZE / nome if isinstance(nome, str) else None
        if candidato_meta in candidatos:
            print(f"{ano}: usando versão registrada em metadata/bronze_{ano}.json")
            return candidato_meta

    if len(candidatos) == 1:
        print(f"{ano}: usando único arquivo local: {candidatos[0].name}")
        return candidatos[0]
    opcoes = ", ".join(p.name for p in candidatos)
    raise ValueError(
        f"{ano}: há várias versões locais; defina NOME_ARQUIVO_DIAGNOSTICO. "
        f"Disponíveis: {opcoes}"
    )


def escolher_codificacao(arquivo):
    with arquivo.open("rb") as entrada:
        inicio = entrada.read(4)
    if inicio.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig", "BOM UTF-8 encontrado"
    if inicio.startswith((b"\xff\xfe", b"\xfe\xff")):
        raise ValueError(f"{arquivo.name}: BOM UTF-16 encontrado; validar o formato antes da leitura")
    return CODIFICACAO_PADRAO, "Sem BOM; codificação presumida, não detectada automaticamente"


def diagnosticar_csv(ano, *, nome_arquivo=None):
    arquivo = selecionar_csv_local(ano, nome_arquivo=nome_arquivo)
    encoding, criterio_encoding = escolher_codificacao(arquivo)
    print(f"Diagnóstico de {arquivo.name}; encoding={encoding} ({criterio_encoding})")

    total_estrutura = 0
    linhas_invalidas = 0
    exemplos_invalidos = []
    try:
        with arquivo.open("r", encoding=encoding, errors="strict", newline="") as entrada:
            leitor = csv.reader(entrada, delimiter=";", strict=True)
            cabecalho = next(leitor)
            if not cabecalho:
                raise ValueError(f"{ano}: cabeçalho CSV vazio")
            for linha in leitor:
                total_estrutura += 1
                if len(linha) != len(cabecalho):
                    linhas_invalidas += 1
                    if len(exemplos_invalidos) < 5:
                        exemplos_invalidos.append((leitor.line_num, len(linha)))
    except (UnicodeError, csv.Error) as erro:
        raise RuntimeError(f"{ano}: erro de leitura/estrutura CSV: {erro}") from erro

    if total_estrutura == 0:
        raise ValueError(f"{ano}: arquivo não contém registros")
    if linhas_invalidas:
        raise RuntimeError(
            f"{ano}: {linhas_invalidas} linhas têm número incorreto de campos. "
            f"Exemplos (linha física, campos): {exemplos_invalidos}. "
            "CSV original preservado; não prosseguir sem investigação."
        )

    total_linhas = 0
    vazios = pd.Series(0, index=cabecalho, dtype="int64")
    try:
        for bloco in pd.read_csv(
            arquivo, sep=";", encoding=encoding, encoding_errors="strict",
            dtype=str, keep_default_na=False, chunksize=5000,
            low_memory=False, on_bad_lines="error", skip_blank_lines=False,
        ):
            contagem = bloco.apply(lambda coluna: coluna.str.strip().eq("").sum())
            vazios = vazios.add(contagem, fill_value=0)
            total_linhas += len(bloco)
    except (UnicodeError, pd.errors.ParserError) as erro:
        raise RuntimeError(f"{ano}: Pandas não conseguiu interpretar o CSV: {erro}") from erro

    if total_linhas != total_estrutura:
        raise RuntimeError(f"{ano}: contagens divergentes: csv={total_estrutura}; pandas={total_linhas}")

    quadro = pd.DataFrame({
        "Coluna": vazios.index,
        "Valores vazios": vazios.astype(int).values,
    })
    quadro["Percentual vazio"] = (quadro["Valores vazios"] / total_linhas * 100).round(4)
    foco = quadro.set_index("Coluna").reindex(CAMPOS_INTERESSE)
    campos = {
        nome: {
            "valores_vazios": int(foco.loc[nome, "Valores vazios"]),
            "percentual_vazio": float(foco.loc[nome, "Percentual vazio"]),
        }
        for nome in CAMPOS_INTERESSE if nome in set(quadro["Coluna"])
    }
    relatorio = {
        "schema_versao": 2,
        "ano": ano, "arquivo": arquivo.name,
        "total_registros": total_linhas,
        "total_colunas": len(quadro),
        "linhas_estrutura_invalida": linhas_invalidas,
        "codificacao_utilizada": encoding,
        "criterio_codificacao": criterio_encoding,
        "data_analise_utc": datetime.now(timezone.utc).isoformat(),
        "campos_analisados": campos,
    }
    salvar_json_atomico(METADATA / f"qualidade_{ano}.json", relatorio)
    print(f"Relatório salvo: metadata/qualidade_{ano}.json")
    print(f"Registros: {total_linhas:,}; colunas: {len(quadro)}; linhas inválidas: {linhas_invalidas}")
    display(foco)
    return relatorio


# Mantenha nome_arquivo=None para usar a versão da lista de URLs ou a seleção local controlada.
# Execute o diagnóstico somente quando desejar; para arquivos grandes, pode levar alguns minutos.
relatorio_qualidade = diagnosticar_csv(
    ANO_DIAGNOSTICO, nome_arquivo=NOME_ARQUIVO_DIAGNOSTICO
)

Diagnóstico de INFLUD24-23-03-2026.csv; encoding=latin-1 (Sem BOM; codificação presumida, não detectada automaticamente)
Relatório salvo: metadata/qualidade_2024.json
Registros: 267,986; colunas: 194; linhas inválidas: 0


,Valores vazios,Percentual vazio
Coluna,,
NU_NOTIFIC,0,0.0000
DT_NOTIFIC,0,0.0000
DT_SIN_PRI,0,0.0000
CS_SEXO,0,0.0000
SG_UF_NOT,6,0.0022
CLASSI_FIN,9088,3.3912
EVOLUCAO,14909,5.5634
HOSPITAL,5803,2.1654
UTI,26295,9.8121


verifica a qualidade e integridade dos arquivos

In [62]:
import json
import hashlib
import pandas as pd

resumo_geral = []
resultados_integridade = []

for ano in range(2019, 2027):
    # 1. Carregar relatório de qualidade
    caminho = METADATA / f"qualidade_{ano}.json"

    if not caminho.exists():
        print(f"{ano}: relatório de qualidade não encontrado")
        continue

    with caminho.open(encoding="utf-8") as f:
        dados = json.load(f)

    resumo_geral.append({
        "Ano": ano,
        "Ano registrado": dados.get("ano"),
        "Arquivo": dados.get("arquivo"),
        "Registros": dados.get("total_registros"),
        "Colunas": dados.get("total_colunas"),
        "Linhas inválidas": dados.get(
            "linhas_estrutura_invalida",
            dados.get("linhas_invalidas")
        )
    })

    # 2. Verificar integridade do arquivo original
    arquivo = selecionar_csv_local(ano)

    caminho_meta = METADATA / f"bronze_{ano}.json"

    with caminho_meta.open(encoding="utf-8") as f:
        metadados = json.load(f)

    sha256 = hashlib.sha256()

    with arquivo.open("rb") as f:
        for bloco in iter(
            lambda: f.read(4 * 1024 * 1024), b""
        ):
            sha256.update(bloco)

    integridade = (
        metadados.get("ano") == ano
        and metadados.get("arquivo") == arquivo.name
        and metadados.get("tamanho_bytes") == arquivo.stat().st_size
        and metadados.get("sha256") == sha256.hexdigest()
    )

    resultados_integridade.append({
        "Ano": ano,
        "Arquivo": arquivo.name,
        "Integridade": "OK" if integridade else "DIVERGÊNCIA"
    })

# 3. Exibir relatórios
df_resumo = pd.DataFrame(resumo_geral)
df_integridade = pd.DataFrame(resultados_integridade)

print("RESUMO GERAL DA CAMADA BRONZE")
display(df_resumo)

print("\nVERIFICAÇÃO DE INTEGRIDADE")
display(df_integridade)

# 4. Resultado final
if (
    len(df_resumo) == 8
    and len(df_integridade) == 8
    and (df_resumo["Ano"] == df_resumo["Ano registrado"]).all()
    and (df_resumo["Linhas inválidas"] == 0).all()
    and (df_integridade["Integridade"] == "OK").all()
):
    print("VALIDAÇÃO LOCAL DA BRONZE CONCLUÍDA COM SUCESSO!")
else:
    print("ATENÇÃO: existem pendências na validação da Bronze.")

RESUMO GERAL DA CAMADA BRONZE


,Ano,Ano registrado,Arquivo,Registros,Colunas,Linhas inválidas
0,2019,2019,INFLUD19-23-03-2026.csv,48941,194,0
1,2020,2020,INFLUD20-23-03-2026.csv,1206920,194,0
2,2021,2021,INFLUD21-23-03-2026.csv,1745672,194,0
3,2022,2022,INFLUD22-23-03-2026.csv,560577,194,0
4,2023,2023,INFLUD23-23-03-2026.csv,279453,194,0
5,2024,2024,INFLUD24-23-03-2026.csv,267986,194,0
6,2025,2025,INFLUD25-16-11-2026.csv,336435,194,0
7,2026,2026,INFLUD26-16-11-2026.csv,248899,194,0



VERIFICAÇÃO DE INTEGRIDADE


,Ano,Arquivo,Integridade
0,2019,INFLUD19-23-03-2026.csv,OK
1,2020,INFLUD20-23-03-2026.csv,OK
2,2021,INFLUD21-23-03-2026.csv,OK
3,2022,INFLUD22-23-03-2026.csv,OK
4,2023,INFLUD23-23-03-2026.csv,OK
5,2024,INFLUD24-23-03-2026.csv,OK
6,2025,INFLUD25-16-11-2026.csv,OK
7,2026,INFLUD26-16-11-2026.csv,OK


VALIDAÇÃO LOCAL DA BRONZE CONCLUÍDA COM SUCESSO!


## 8. Antes de publicar no GitHub

1. Confira `git status`: os CSVs originais e temporários de `data/bronze/` não devem ser enviados.
2. Versione o notebook, README e JSONs de metadados agregados.
3. Documente a versão do CSV utilizada na futura Silver.
4. Revise manualmente as URLs oficiais, especialmente 2025 e 2026, quando houver nova publicação.
5. Execute novamente os testes no Linux e no Windows; cada máquina usa sua própria `.venv`.

**Não existe consulta à API CKAN neste notebook.** O texto `ckan.saude.gov.br` pode continuar aparecendo apenas nas URLs de objetos CSV publicados no armazenamento S3 oficial.

**Nota:** ETag não é checksum criptográfico; SHA-256 identifica bytes de um arquivo local, mas sua comparação com a fonte exige referência confiável. A Bronze preserva o arquivo original sem fazer correções de conteúdo.
